# DRI and KPI Delta Queries

In [ ]:
import sys, pathlib
repo_root = pathlib.Path.cwd()
for _ in range(4):
    if (repo_root / "src").exists():
        break
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
from sqlalchemy import text
from src.config.db_config import get_engine

engine = get_engine()

## Decision Reliability Index (DRI)
Example formula (adjust weights as needed):
- `dq_score = max(0, 1 - anomaly_rate - violation_rate)`
- `dri = 0.6 * dq_score + 0.4 * user_trust`

Assumes a `user_trust` table with columns: `source_table`, `user_group`, `trust_score` (0–1). If absent, use a placeholder constant.

In [ ]:
dri_sql = text(
    """
    with latest_anomaly as (
        select distinct on (source_table, model_name)
            source_table, anomaly_rate, created_at
        from dq.anomaly_metrics
        order by source_table, model_name, created_at desc
    ),
    latest_validation as (
        select distinct on (source_table, rule_name)
            source_table, violation_rate, created_at
        from dq.validation_metrics
        order by source_table, rule_name, created_at desc
    ),
    agg_validation as (
        select source_table, avg(violation_rate) as violation_rate
        from latest_validation
        group by source_table
    ),
    agg_anomaly as (
        select source_table, avg(anomaly_rate) as anomaly_rate
        from latest_anomaly
        group by source_table
    ),
    trust as (
        -- replace with a real user trust table if available
        select source_table, 0.8::numeric as trust_score
        from (select distinct source_table from dq.anomaly_metrics) t
    )
    select
        coalesce(a.source_table, v.source_table, t.source_table) as source_table,
        coalesce(a.anomaly_rate, 0) as anomaly_rate,
        coalesce(v.violation_rate, 0) as violation_rate,
        coalesce(t.trust_score, 0.5) as user_trust,
        greatest(0, 1 - coalesce(a.anomaly_rate,0) - coalesce(v.violation_rate,0)) as dq_score,
        0.6 * greatest(0, 1 - coalesce(a.anomaly_rate,0) - coalesce(v.violation_rate,0)) + 0.4 * coalesce(t.trust_score,0.5) as dri
    from agg_anomaly a
    full outer join agg_validation v on a.source_table = v.source_table
    full outer join trust t on coalesce(a.source_table, v.source_table) = t.source_table
    """
)

dri_df = pd.read_sql(dri_sql, engine)
dri_df

## KPI deltas (raw vs processed)
Example KPIs: revenue and average order value (AOV). Adjust column names as needed.

In [ ]:
kpi_sql = text(
    """
    with raw_kpi as (
      select
        'ecommerce' as ds,
        sum(price * quantity) as revenue,
        avg(price * quantity) as aov
      from raw.ecommerce_transactions
      union all
      select 'online_retail', sum(unit_price * quantity), avg(unit_price * quantity)
      from raw.online_retail
    ),
    proc_kpi as (
      select
        'ecommerce' as ds,
        sum(price * quantity) as revenue,
        avg(price * quantity) as aov
      from processed.ecommerce_transactions
      union all
      select 'online_retail', sum(unit_price * quantity), avg(unit_price * quantity)
      from processed.online_retail
    )
    select
      r.ds,
      r.revenue as raw_revenue,
      p.revenue as processed_revenue,
      p.revenue - r.revenue as revenue_delta,
      r.aov as raw_aov,
      p.aov as processed_aov,
      p.aov - r.aov as aov_delta
    from raw_kpi r
    join proc_kpi p on r.ds = p.ds
    """
)

kpi_df = pd.read_sql(kpi_sql, engine)
kpi_df